In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

DATA_PATH = Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(DATA_PATH)


In [8]:
# Remove the customer identifier
df = df.drop(columns=["customerID"])

# Binary encoding
binary_maps = {
    "gender": {"Female": 0, "Male": 1},
    "Partner": {"No": 0, "Yes": 1},
    "Dependents": {"No": 0, "Yes": 1},
    "PhoneService": {"No": 0, "Yes": 1},
    "PaperlessBilling": {"No": 0, "Yes": 1},
    "Churn": {"No": 0, "Yes": 1},
}

for col, mapping in binary_maps.items():
    df[col] = df[col].map(mapping)

# One-hot encode remaining categorical variables
categorical_cols = [
    "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaymentMethod"
]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

display(df.head())
print("Final pre-split shape:", df.shape)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,False,False,False,True,False
1,1,0,0,0,34,1,0,56.95,1889.5,0,...,False,False,False,False,False,True,False,False,False,True
2,1,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,False,False,False,True
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,True,False,False,False,False,True,False,False,False,False
4,0,0,0,0,2,1,1,70.70,151.65,1,...,False,False,False,False,False,False,False,False,True,False


Final pre-split shape: (7043, 31)


In [9]:
# Convert TotalCharges from string to numeric
# Blank strings become NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# The 11 blank values correspond to customers with tenure = 0,
# so fill them with 0 rather than dropping the rows.
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
display(y_train.value_counts(normalize=True))

print("Test target distribution:")
display(y_test.value_counts(normalize=True))


Train shape: (5634, 30)
Test shape: (1409, 30)

Train target distribution:


Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Test target distribution:


Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64

In [11]:
# Standardize continuous variables using training data only.
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])


### Feature engineering and multicollinearity cleanup

The original notebook created `TotalAddons` from the six `_Yes` service indicators and then used VIF to identify redundant features. It removed:

- the six `"No internet service"` indicators,
- the six raw addon `_Yes` indicators after creating `TotalAddons`,
- `MultipleLines_No phone service`,
- `TotalCharges`.

This was done to reduce encoding artifacts and multicollinearity before modeling.


In [12]:
addon_yes_cols = [
    "OnlineSecurity_Yes", "OnlineBackup_Yes", "DeviceProtection_Yes",
    "TechSupport_Yes", "StreamingTV_Yes", "StreamingMovies_Yes"
]

X_train["TotalAddons"] = X_train[addon_yes_cols].sum(axis=1)
X_test["TotalAddons"] = X_test[addon_yes_cols].sum(axis=1)

print("TotalAddons distribution:")
display(X_train["TotalAddons"].value_counts().sort_index())

print("Churn rate by TotalAddons:")
display(
    pd.concat([X_train["TotalAddons"], y_train], axis=1)
      .groupby("TotalAddons")["Churn"]
      .mean()
)


TotalAddons distribution:


TotalAddons
0    1757
1     766
2     832
3     885
4     695
5     464
6     235
Name: count, dtype: int64

Churn rate by TotalAddons:


TotalAddons
0    0.209448
1    0.458225
2    0.361779
3    0.277966
4    0.230216
5    0.125000
6    0.046809
Name: Churn, dtype: float64

In [13]:
# Initial correlation check
display(X_train[["tenure", "MonthlyCharges", "TotalCharges"]].corr())


,tenure,MonthlyCharges,TotalCharges
tenure,1.000000,0.256700,0.829698
MonthlyCharges,0.256700,1.000000,0.654117
TotalCharges,0.829698,0.654117,1.000000


In [14]:
# VIF requires numeric input
X_train = X_train.astype(float)
X_test = X_test.astype(float)

from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif = pd.DataFrame({
        "feature": X.columns,
        "VIF": [
            variance_inflation_factor(X.values, i)
            for i in range(X.shape[1])
        ]
    })
    return vif.sort_values("VIF", ascending=False)

print("Initial VIF:")
display(calculate_vif(X_train).head(15))


Initial VIF:


c:\Users\TIAA USER\ml-notebook\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
12,InternetService_No,inf
14,OnlineSecurity_Yes,inf
13,OnlineSecurity_No internet service,inf
18,DeviceProtection_Yes,inf
30,TotalAddons,inf
24,StreamingMovies_Yes,inf
23,StreamingMovies_No internet service,inf
22,StreamingTV_Yes,inf
21,StreamingTV_No internet service,inf
20,TechSupport_Yes,inf


In [15]:
# Remove duplicated/no-longer-needed encoded features identified in the original analysis.
redundant_cols = [
    "OnlineSecurity_No internet service",
    "OnlineBackup_No internet service",
    "DeviceProtection_No internet service",
    "TechSupport_No internet service",
    "StreamingTV_No internet service",
    "StreamingMovies_No internet service",
]

X_train = X_train.drop(columns=redundant_cols)
X_test = X_test.drop(columns=redundant_cols)

X_train = X_train.drop(columns=addon_yes_cols)
X_test = X_test.drop(columns=addon_yes_cols)

X_train = X_train.drop(columns=["MultipleLines_No phone service"])
X_test = X_test.drop(columns=["MultipleLines_No phone service"])

X_train = X_train.drop(columns=["TotalCharges"])
X_test = X_test.drop(columns=["TotalCharges"])

print("Final feature count:", X_train.shape[1])
print("\nFinal VIF:")
display(calculate_vif(X_train).head(15))


Final feature count: 17

Final VIF:


,feature,VIF
7,MonthlyCharges,11.595051
5,PhoneService,9.707248
9,InternetService_Fiber optic,7.342117
16,TotalAddons,6.488581
10,InternetService_No,5.614565
12,Contract_Two year,3.392759
6,PaperlessBilling,2.861071
14,PaymentMethod_Electronic check,2.836946
2,Partner,2.800868
4,tenure,2.676890
